# 04 – Functions

Functions are the primary unit of code reuse in Python.

Topics covered:
- Defining and calling functions
- Parameters: positional, keyword, default, `*args`, `**kwargs`
- Return values
- Scope (LEGB rule)
- `lambda` — anonymous functions
- `map()`, `filter()`, `reduce()`
- Docstrings and type hints

## 1. Defining Functions

In [1]:
# Basic function
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))

# Function with no return → implicitly returns None
def log(message):
    print(f"[LOG] {message}")

result = log("started")
print(result)  # None

Hello, Alice!
[LOG] started
None


In [2]:
# Multiple return values (returns a tuple)
def min_max(numbers):
    return min(numbers), max(numbers)

lo, hi = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(f"min={lo}, max={hi}")

min=1, max=9


## 2. Parameter Types

```
def func(pos, /, normal, *, kw_only, **kwargs):
```

| Kind | Description |
|------|-------------|
| Positional | Must be passed in order |
| Default | Optional, has a fallback |
| `*args` | Captures extra positional args as a tuple |
| Keyword-only | Must be passed by name (after `*`) |
| `**kwargs` | Captures extra keyword args as a dict |

In [3]:
# Positional and default parameters
def power(base, exponent=2):
    return base ** exponent

print(power(3))      # 9  (uses default)
print(power(3, 3))   # 27 (positional)
print(power(base=2, exponent=10))  # 1024 (keyword)

9
27
1024


In [4]:
# IMPORTANT: mutable default argument bug
# WRONG:
def append_wrong(item, lst=[]):
    lst.append(item)
    return lst

print(append_wrong(1))  # [1]
print(append_wrong(2))  # [1, 2] ← list persists across calls!

# CORRECT: use None as default
def append_correct(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(append_correct(1))  # [1]
print(append_correct(2))  # [2]  ← fresh each time

[1]
[1, 2]
[1]
[2]


In [5]:
# *args — variable positional arguments
def add_all(*numbers):
    return sum(numbers)

print(add_all(1, 2, 3))       # 6
print(add_all(10, 20, 30, 40))  # 100

# Unpacking with *
nums = [1, 2, 3, 4, 5]
print(add_all(*nums))  # unpack list as positional args

6
100
15


In [6]:
# **kwargs — variable keyword arguments
def create_profile(**info):
    for key, value in info.items():
        print(f"  {key}: {value}")

create_profile(name="Alice", age=30, role="Engineer")

# Combining all types
def full_example(a, b, *args, keyword_only=True, **kwargs):
    print(a, b, args, keyword_only, kwargs)

full_example(1, 2, 3, 4, keyword_only=False, x=10, y=20)

  name: Alice
  age: 30
  role: Engineer
1 2 (3, 4) False {'x': 10, 'y': 20}


## 3. Scope — LEGB Rule

Python looks up names in this order:
**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

In [7]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print("inner:", x)    # local
    inner()
    print("outer:", x)        # enclosing

outer()
print("global:", x)           # global

inner: local
outer: enclosing
global: global


In [8]:
# global keyword — modify a global variable inside a function
count = 0

def increment():
    global count
    count += 1

increment()
increment()
print(count)  # 2

# nonlocal — modify an enclosing scope variable
def make_counter():
    n = 0
    def counter():
        nonlocal n
        n += 1
        return n
    return counter

c = make_counter()
print(c(), c(), c())  # 1 2 3

2
1 2 3


### The late-binding closure gotcha

A closure captures the enclosing **variable**, not the value it held at closure-creation time.
If a loop creates several closures over the same loop variable, every closure shares that one
variable — and by the time any of them are *called*, the loop has already finished, so they all
see the loop variable's *final* value. This is a real, common bug, reproduced below before fixing
it.

In [9]:
# BUGGY: every lambda closes over the same variable `i`, not its value at creation time
buggy_funcs = [lambda: i for i in range(3)]
print("buggy (all see final i):", [f() for f in buggy_funcs])   # expect [2, 2, 2], not [0, 1, 2]

# FIX 1: default-argument trick — default values ARE evaluated at def time, capturing i's value then
fixed_funcs_default = [lambda i=i: i for i in range(3)]
print("fixed via default arg:  ", [f() for f in fixed_funcs_default])

# FIX 2: a factory function — each call creates a genuinely new local scope/variable
def make_returner(value):
    return lambda: value

fixed_funcs_factory = [make_returner(i) for i in range(3)]
print("fixed via factory func: ", [f() for f in fixed_funcs_factory])

buggy (all see final i): [2, 2, 2]
fixed via default arg:   [0, 1, 2]
fixed via factory func:  [0, 1, 2]


## 7. From-scratch: what `@decorator` desugars to

`@decorator` above a `def` is syntactic sugar. It is exactly equivalent to writing the plain
function, then rebinding its name to `decorator(original_function)` — no more, no less. Built
below the long way first (no `@`), then rewritten with `@` to show they produce identical
behavior.

In [10]:
import time

def slow_add(a, b):
    time.sleep(0.01)
    return a + b

# The manual way: what "@timing_decorator" desugars to, written out in full
def timing_decorator(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

# No '@' syntax at all — just an ordinary function call that rebinds the name
slow_add = timing_decorator(slow_add)
print("manual desugared call:", slow_add(2, 3))

slow_add took 0.0101s
manual desugared call: 5


In [11]:
# Same decorator, now applied with the shorthand — identical behavior, less boilerplate
@timing_decorator
def slow_multiply(a, b):
    time.sleep(0.01)
    return a * b

print("'@' syntax call:      ", slow_multiply(4, 5))

# functools.wraps — a real gotcha with hand-written decorators: the wrapper shadows
# the original function's __name__/__doc__ unless explicitly preserved
print("wrapper hides identity:", slow_multiply.__name__)  # 'wrapper', not 'slow_multiply' — a real bug for introspection/debugging

from functools import wraps

def timing_decorator_fixed(func):
    @wraps(func)   # copies __name__, __doc__, etc. from func onto wrapper
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timing_decorator_fixed
def slow_divide(a, b):
    time.sleep(0.01)
    return a / b

print("identity preserved:    ", slow_divide.__name__)

slow_multiply took 0.0101s
'@' syntax call:       20
wrapper hides identity: wrapper
identity preserved:     slow_divide


## 4. Lambda Functions

Anonymous single-expression functions. Syntax: `lambda args: expression`  
Best used inline, not assigned to a variable.

In [12]:
# Equivalent definitions
def square(x): return x ** 2
sq = lambda x: x ** 2

print(square(5), sq(5))  # 25 25

# Multi-argument lambda
add = lambda x, y: x + y
print(add(3, 4))  # 7

# Lambda as a sort key — very common pattern
students = [("Alice", 92), ("Bob", 88), ("Charlie", 95)]
students.sort(key=lambda s: s[1], reverse=True)  # sort by score
print(students)

25 25
7
[('Charlie', 95), ('Alice', 92), ('Bob', 88)]


## 5. `map()`, `filter()`, `reduce()`

Functional programming tools for transforming and filtering sequences.

In [13]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map(function, iterable) — apply function to every element
squares = list(map(lambda x: x**2, nums))
print("squares:", squares)

# filter(function, iterable) — keep elements where function returns True
evens = list(filter(lambda x: x % 2 == 0, nums))
print("evens:", evens)

# reduce — fold a sequence to a single value
from functools import reduce
product = reduce(lambda acc, x: acc * x, nums)
print("product:", product)

squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
evens: [2, 4, 6, 8, 10]
product: 3628800


In [14]:
# Practical: transform strings
names = ["alice", "bob", "charlie"]
upper = list(map(str.upper, names))
print(upper)

long_names = list(filter(lambda n: len(n) > 3, names))
print(long_names)

# NOTE: list comprehensions are often cleaner
upper_comp = [n.upper() for n in names]
long_comp  = [n for n in names if len(n) > 3]
print(upper_comp, long_comp)

['ALICE', 'BOB', 'CHARLIE']
['alice', 'charlie']
['ALICE', 'BOB', 'CHARLIE'] ['alice', 'charlie']


## 6. Docstrings and Type Hints

Good practice for any function that others (or future you) will read.

In [15]:
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """
    Calculate Body Mass Index (BMI).

    Args:
        weight_kg: Weight in kilograms.
        height_m:  Height in metres.

    Returns:
        BMI value rounded to 2 decimal places.

    Raises:
        ValueError: If height_m is zero or negative.
    """
    if height_m <= 0:
        raise ValueError("height_m must be positive")
    return round(weight_kg / height_m ** 2, 2)

print(calculate_bmi(70, 1.75))    # 22.86
help(calculate_bmi)               # shows the docstring

22.86
Help on function calculate_bmi in module __main__:

calculate_bmi(weight_kg: float, height_m: float) -> float
    Calculate Body Mass Index (BMI).

    Args:
        weight_kg: Weight in kilograms.
        height_m:  Height in metres.

    Returns:
        BMI value rounded to 2 decimal places.

    Raises:
        ValueError: If height_m is zero or negative.



In [16]:
# Type hints don't enforce at runtime — they are for readability & static analysis (mypy)
from typing import Optional, Union, List

def process(items: List[int], limit: Optional[int] = None) -> List[int]:
    if limit is not None:
        items = items[:limit]
    return [x * 2 for x in items]

print(process([1, 2, 3, 4, 5], limit=3))  # [2, 4, 6]

[2, 4, 6]


## Quick Summary

| Concept | Key Takeaway |
|---------|-------------|
| `def` | Define a function — use descriptive names |
| Default args | Use `None` for mutable defaults |
| `*args` | Catch extra positional args as a tuple |
| `**kwargs` | Catch extra keyword args as a dict |
| LEGB | Python looks Local → Enclosing → Global → Built-in |
| `lambda` | One-liner anonymous function for use inline |
| `map/filter` | Apply/filter; prefer comprehensions when clearer |
| Docstrings | Document intent; type hints aid editors and linters |

**Next →** [05 – Modules & Packages](../05-modules-packages/)